# CBIR Pipeline - Preprocessing Method 2

Output artifacts are saved as `index/cbir_method2_index.npz` and `index/cbir_method2_meta.json`.

In [ ]:
from __future__ import annotations
from mediapipe.tasks.python import vision as mp_vision
from pathlib import Path
from typing import Any
import json
import random

import cv2
import numpy as np
from numpy.typing import NDArray
import face_recognition
import mediapipe as mp
import urllib.request

ImageArray = NDArray[np.uint8]
FloatArray = NDArray[np.float32]

project_root: Path = Path.cwd()
if not (project_root / "data").exists():
    project_root = project_root.parent

FACE_DIR: Path = project_root / "data" / "face"
INDEX_DIR: Path = project_root / "index"
INDEX_DIR.mkdir(parents=True, exist_ok=True)

CBIR_INDEX_PATH: Path = INDEX_DIR / "cbir_method2_index.npz"
CBIR_META_PATH: Path = INDEX_DIR / "cbir_method2_meta.json"
CBIR_SPLIT_PATH: Path = INDEX_DIR / "cbir_eval_split.json"

OUTPUT_SIZE: tuple[int, int] = (128, 128)
IMAGE_EXTS: set[str] = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

GALLERY_RATIO: float = 0.8
SPLIT_SEED: int = 42
_FACE_DETECTOR = None

print(f"Face folder: {FACE_DIR}")
print(f"CBIR method2 index output: {CBIR_INDEX_PATH}")
print(f"CBIR method2 meta output : {CBIR_META_PATH}")
print(f"Shared split output      : {CBIR_SPLIT_PATH}")

Face folder: c:\Users\jaft9\School\AttSystem\data\face
CBIR method2 index output: c:\Users\jaft9\School\AttSystem\index\cbir_method2_index.npz
CBIR method2 meta output : c:\Users\jaft9\School\AttSystem\index\cbir_method2_meta.json
Shared split output      : c:\Users\jaft9\School\AttSystem\index\cbir_eval_split.json


In [ ]:
def get_face_detector():
    global _FACE_DETECTOR
    if _FACE_DETECTOR is not None:
        return _FACE_DETECTOR
        
    model_path = project_root / "models" / "blaze_face_short_range.tflite"
    if not model_path.exists():
        model_path.parent.mkdir(parents=True, exist_ok=True)
        print("Downloading BlazeFace model...")
        url = "https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_short_range/float16/1/blaze_face_short_range.tflite"
        urllib.request.urlretrieve(url, str(model_path))
        
    options = mp_vision.FaceDetectorOptions(
        base_options=mp.tasks.BaseOptions(model_asset_path=str(model_path)),
        min_detection_confidence=0.5
    )
    _FACE_DETECTOR = mp_vision.FaceDetector.create_from_options(options)
    return _FACE_DETECTOR


def detect_face_roi(bgr: ImageArray) -> ImageArray:
    face_detector = get_face_detector()
    
    h, w = bgr.shape[:2]
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    results = face_detector.detect(mp_image)
    
    if not results.detections:
        side: int = int(min(h, w) * 0.75)
        cx, cy = w // 2, h // 2
        x1: int = max(0, cx - side // 2)
        y1: int = max(0, cy - side // 2)
        x2: int = min(w, x1 + side)
        y2: int = min(h, y1 + side)
        return rgb[y1:y2, x1:x2].astype(np.uint8, copy=False)
        
    best_detection = max(
        results.detections,
        key=lambda d: d.bounding_box.width * d.bounding_box.height
    )
    bboxC = best_detection.bounding_box
    x = int(bboxC.origin_x)
    y = int(bboxC.origin_y)
    fw = int(bboxC.width)
    fh = int(bboxC.height)
    
    pad_w: int = int(fw * 0.2)
    pad_h: int = int(fh * 0.2)

    x1: int = max(0, x - pad_w)
    y1: int = max(0, y - pad_h)
    x2: int = min(w, x + fw + pad_w)
    y2: int = min(h, y + fh + pad_h)

    return rgb[y1:y2, x1:x2].astype(np.uint8, copy=False)

def preprocess_roi_method2(roi_rgb: ImageArray) -> ImageArray:
    # 1. Bilateral filter for initial denoising
    denoised = cv2.bilateralFilter(roi_rgb, d=5, sigmaColor=50, sigmaSpace=50)

    # 2. Convert to LAB space for CLAHE
    lab = cv2.cvtColor(denoised, cv2.COLOR_RGB2LAB)
    l_c, a_c, b_c = cv2.split(lab)
    
    # 3. Apply CLAHE on L-channel
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l_c = clahe.apply(l_c)
    
    # 4. Merge back to RGB
    lab_enhanced = cv2.merge([l_c, a_c, b_c])
    enhanced_roi = cv2.cvtColor(lab_enhanced, cv2.COLOR_LAB2RGB)
    
    # 5. Resize to output size
    resized = cv2.resize(enhanced_roi, OUTPUT_SIZE, interpolation=cv2.INTER_CUBIC)
    return resized.astype(np.uint8, copy=False)

def extract_embedding_from_face(face_rgb: ImageArray) -> FloatArray | None:
    h, w = face_rgb.shape[:2]
    known_face_locations = [(0, w, h, 0)]
    encodings: list[Any] = face_recognition.face_encodings(face_rgb, known_face_locations=known_face_locations)
    if len(encodings) == 0:
        return None

    emb = np.array(encodings[0], dtype=np.float32)
    return emb

def build_identity_split(
    face_dir: Path,
    gallery_ratio: float = GALLERY_RATIO,
    seed: int = SPLIT_SEED,
    split_path: Path = CBIR_SPLIT_PATH,
    overwrite: bool = False,
    min_test_per_identity: int = 1,
    min_gallery_per_identity: int = 1,
    min_files_for_split: int = 3,
) -> dict[str, dict[str, list[str]]]:
    if split_path.exists() and not overwrite:
        with split_path.open("r", encoding="utf-8") as f:
            split_data = json.load(f)
        return split_data

    if not face_dir.exists() or not face_dir.is_dir():
        raise FileNotFoundError(f"Missing dataset folder: {face_dir}")

    rng = random.Random(seed)
    split_data: dict[str, dict[str, list[str]]] = {}

    person_dirs: list[Path] = sorted([p for p in face_dir.iterdir() if p.is_dir()])
    if not person_dirs:
        raise RuntimeError("No person folders found in data/face.")

    for person_dir in person_dirs:
        files: list[Path] = sorted([f for f in person_dir.iterdir() if f.is_file() and f.suffix.lower() in IMAGE_EXTS])
        if not files:
            continue

        names = [f.name for f in files]
        shuffled = names[:]
        rng.shuffle(shuffled)

        if len(shuffled) < min_files_for_split:
            gallery = shuffled[:]
            test = []
        else:
            desired_gallery = int(round(len(shuffled) * gallery_ratio))
            max_gallery = len(shuffled) - min_test_per_identity
            desired_gallery = min(desired_gallery, max_gallery)
            desired_gallery = max(min_gallery_per_identity, desired_gallery)

            gallery = shuffled[:desired_gallery]
            test = shuffled[desired_gallery:]

        split_data[person_dir.name] = {
            "gallery": gallery,
            "test": test,
        }

    with split_path.open("w", encoding="utf-8") as f:
        json.dump(
            {
                "seed": seed,
                "gallery_ratio": gallery_ratio,
                "identities": split_data,
            },
            f,
            indent=2,
        )

    return {
        name: payload
        for name, payload in split_data.items()
    } if isinstance(next(iter(split_data.values()), {}), dict) else split_data

def _normalize_split_payload(split_payload: dict[str, Any]) -> dict[str, dict[str, list[str]]]:
    if "identities" in split_payload and isinstance(split_payload["identities"], dict):
        identities = split_payload["identities"]
    else:
        identities = split_payload

    normalized: dict[str, dict[str, list[str]]] = {}
    for person_name, payload in identities.items():
        if not isinstance(payload, dict):
            continue
        gallery = [str(x) for x in payload.get("gallery", [])]
        test = [str(x) for x in payload.get("test", [])]
        normalized[str(person_name)] = {"gallery": gallery, "test": test}
    return normalized

def build_cbir_gallery_method2(
    face_dir: Path,
    split_data: dict[str, dict[str, list[str]]],
) -> tuple[FloatArray, NDArray[np.int32], list[str], list[str]]:
    if not face_dir.exists() or not face_dir.is_dir():
        raise FileNotFoundError(f"Missing dataset folder: {face_dir}")

    person_dirs: list[Path] = sorted([p for p in face_dir.iterdir() if p.is_dir()])
    if not person_dirs:
        raise RuntimeError("No person folders found in data/face.")

    embeddings: list[FloatArray] = []
    label_ids: list[int] = []
    label_names: list[str] = []
    image_paths: list[str] = []

    total_gallery_images: int = 0
    kept_images: int = 0

    for label_id, person_dir in enumerate(person_dirs):
        files: list[Path] = sorted([f for f in person_dir.iterdir() if f.is_file() and f.suffix.lower() in IMAGE_EXTS])
        if not files:
            continue

        selected_names = set(split_data.get(person_dir.name, {}).get("gallery", []))
        if selected_names:
            files = [f for f in files if f.name in selected_names]

        if not files:
            print(f"{person_dir.name}: no gallery images selected, skipping")
            continue

        label_names.append(person_dir.name)
        person_kept: int = 0

        for f in files:
            total_gallery_images += 1
            bgr = cv2.imread(str(f), cv2.IMREAD_COLOR)
            if bgr is None:
                continue

            roi = detect_face_roi(bgr)
            if roi.size == 0:
                continue

            sample = preprocess_roi_method2(roi)
            emb = extract_embedding_from_face(sample)
            if emb is None:
                continue

            embeddings.append(emb)
            label_ids.append(label_id)
            image_paths.append(str(f))
            person_kept += 1
            kept_images += 1

        print(f"{person_dir.name}: indexed {person_kept}/{len(files)} gallery images")

    if len(embeddings) == 0:
        raise RuntimeError("No usable face embeddings were produced from selected gallery images.")

    print(f"Total gallery read: {total_gallery_images}, indexed: {kept_images}")

    emb_np: FloatArray = np.vstack(embeddings).astype(np.float32)
    label_np: NDArray[np.int32] = np.array(label_ids, dtype=np.int32)
    return emb_np, label_np, label_names, image_paths

In [9]:
def save_cbir_method2_index() -> None:
    split_payload = build_identity_split(
        FACE_DIR,
        gallery_ratio=GALLERY_RATIO,
        seed=SPLIT_SEED,
        split_path=CBIR_SPLIT_PATH,
        overwrite=False,
    )
    split_data = _normalize_split_payload(split_payload)

    embeddings, labels, label_names, image_paths = build_cbir_gallery_method2(FACE_DIR, split_data)

    np.savez_compressed(
        CBIR_INDEX_PATH,
        embeddings=embeddings,
        labels=labels,
        image_paths=np.array(image_paths, dtype=object),
    )

    test_counts = {
        person: len(payload.get("test", []))
        for person, payload in split_data.items()
    }

    meta = {
        "embedding_dim": int(embeddings.shape[1]),
        "num_vectors": int(embeddings.shape[0]),
        "label_names": label_names,
        "distance": "euclidean",
        "source_folder": str(FACE_DIR),
        "preprocess_method": "method2",
        "split_path": str(CBIR_SPLIT_PATH),
        "gallery_ratio": GALLERY_RATIO,
        "split_seed": SPLIT_SEED,
        "test_images_per_identity": test_counts,
    }

    with open(CBIR_META_PATH, "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)

    print("\nCBIR Method 2 pipeline complete")
    print(f"Index saved: {CBIR_INDEX_PATH}")
    print(f"Meta saved : {CBIR_META_PATH}")
    print(f"Split saved: {CBIR_SPLIT_PATH}")
    print(f"Vectors    : {embeddings.shape[0]}")
    print(f"Dim        : {embeddings.shape[1]}")


save_cbir_method2_index()

benjamin: indexed 181/181 gallery images
chern_tak: indexed 216/216 gallery images
chillien: indexed 158/158 gallery images
daniel: indexed 194/194 gallery images
dylan: indexed 148/148 gallery images
han_soon: indexed 138/138 gallery images
harry: indexed 72/72 gallery images
isaac: indexed 40/40 gallery images
jing_ang: indexed 197/197 gallery images
jun_wei: indexed 194/194 gallery images
kang_kai: indexed 216/216 gallery images
marion: indexed 202/202 gallery images
ms_nurul: indexed 79/79 gallery images
qi_xuan: indexed 216/216 gallery images
shuang_quan: indexed 193/193 gallery images
wee_xuan: indexed 173/173 gallery images
xiang_yue: indexed 216/216 gallery images
xu_sheng: indexed 209/209 gallery images
yoke_hong: indexed 214/214 gallery images
yong_kang: indexed 170/170 gallery images
zi_herng: indexed 72/72 gallery images
Total gallery read: 3498, indexed: 3498

CBIR Method 2 pipeline complete
Index saved: c:\Users\jaft9\School\AttSystem\index\cbir_method2_index.npz
Meta sav